# Task 2 Updated: AI Patent Classification Using DISCERN + USPTO AIPD
## Implementation Notebook

**Author:** Edward Jung  
**Date:** March 5, 2026  
**Approach:** Hybrid validation using USPTO AIPD (ML-based) + CPC codes

---

## Overview

This notebook implements **Approach 3** from the documentation:
- **Layer 1:** Map clinical trial firms to patents via DISCERN 2
- **Layer 2:** Classify AI patents using USPTO AIPD (BERT-based ML)
- **Layer 3:** Validate with CPC G06N* codes
- **Output:** Firm-year aggregated dataset with three-tier AI classification

### Expected Performance
- Precision: ~92%
- Recall: ~88%
- Coverage: Patents + pre-grant applications (2000-2021)

## Setup & Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

print("✓ Imports successful")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

✓ Imports successful
Pandas version: 2.3.3
NumPy version: 2.3.5


## Phase 1: Load Data

### 1.1 Load Clinical Trials Dataset

In [2]:
# Load clinical trials data
print("Loading clinical trials dataset...")
clinical_trials = pd.read_csv('clinical_trial_sample.csv', encoding='utf-8-sig')

print(f"✓ Loaded {len(clinical_trials):,} clinical trials")
print(f"  Unique firms (GVKEY): {clinical_trials['gvkey_sponsor'].nunique():,}")
print(f"  Year range: {clinical_trials['start_year'].min()}-{clinical_trials['start_year'].max()}")

# Show sample
clinical_trials.head(3)

Loading clinical trials dataset...
✓ Loaded 9,428 clinical trials
  Unique firms (GVKEY): 673
  Year range: 2008-2021


,nct_id,brief_title,overall_status,sponsor_name,gvkey_sponsor,phase_number,start_date,start_year
0,NCT00175851,Open Label Trial to Study the Long-term Safety...,Withdrawn,UCB Pharma,24454,3,2008-05-01,2008
1,NCT00359632,Study to Evaluate Eye Function in Patients Tak...,Terminated,Pfizer,8530,3,2008-11-01,2008
2,NCT00415155,A Study of LY2181308 in Patients With Advanced...,Withdrawn,Eli Lilly and Company,6730,2,2008-08-01,2008


### 1.2 Load DISCERN 2 Datasets

In [3]:
# Load DISCERN patent applications (this may take 30-60 seconds)
print("Loading DISCERN 2 patent applications (188 MB)...")
discern_patents = pd.read_csv(
    'DISCERN 2_0_1/output_files/csv_files/discern_pat_app_1980_2021.csv',
    dtype={
        'application_id': str,
        'patent_id': str,
        'pgpub_id': str,
        'permno_adj': str,
        'fyear': 'Int64'
    },
    parse_dates=['filing_date', 'published_date']
)

print(f"✓ Loaded {len(discern_patents):,} patent applications")
print(f"  Year range: {discern_patents['fyear'].min()}-{discern_patents['fyear'].max()}")
print(f"  Unique firms (PERMNO): {discern_patents['permno_adj'].nunique():,}")

discern_patents.head(3)

Loading DISCERN 2 patent applications (188 MB)...
✓ Loaded 1,587,310 patent applications
  Year range: 2000-2021
  Unique firms (PERMNO): 3,722


,pgpub_id,application_id,filing_date,assignee_name,patent_id,fyear,published_date,name_std,id_name,sample,permno_adj,name_permno_adj
0,20050138134,9186056,2003-12-22,INTEL CORPORATION,7076578,2003,2005-06-23,INTEL CORP,4943,U,59328,INTEL CORP
1,20040225507,9475961,2002-09-16,GENERAL ELECTRIC COMPANY,7457761,2002,2004-11-11,GEN ELECTR CO,3957,U,12060,GE
2,20040214122,9639198,2002-10-15,"THERMAL DYNAMICS, INC.",6881056,2002,2004-10-28,THERMAL DYNAM CORP,762629,S,80576,THERMADYNE HLDG CORP


In [4]:
# Load PERMNO to GVKEY mapping
print("\nLoading PERMNO-GVKEY mapping...")
permno_gvkey = pd.read_csv(
    'DISCERN 2_0_1/output_files/csv_files/permno_gvkey.csv',
    dtype={
        'gvkey': str,
        'permno_adj': str,
        'fyear1_adjust': 'Int64',
        'fyearn_adjust': 'Int64'
    }
)

print(f"✓ Loaded {len(permno_gvkey):,} PERMNO-GVKEY mappings")
print(f"  Unique GVKEYs: {permno_gvkey['gvkey'].nunique():,}")
print(f"  Unique PERMNOs: {permno_gvkey['permno_adj'].nunique():,}")

permno_gvkey.head(3)


Loading PERMNO-GVKEY mapping...
✓ Loaded 8,208 PERMNO-GVKEY mappings
  Unique GVKEYs: 8,199
  Unique PERMNOs: 8,036


,gvkey,permno_adj,fyear1_adjust,fyearn_adjust,min_y_permno,max_y_permno
0,001010,10006,1980,1984,1980,1984
1,009636,10007,1986,1989,1986,1989
2,012622,10010,1986,1994,1986,1994


### 1.3 Load USPTO AIPD (AI Classifications)

In [5]:
# Load USPTO AIPD predictions (this may take 2-3 minutes, 2.8 GB file)
print("Loading USPTO AIPD AI classifications (2.8 GB)...")
print("This may take 2-3 minutes...")

# Load with optimized dtypes to save memory
aipd_dtypes = {
    'doc_id': str,
    'flag_patent': 'int8',
    'appl_id': str,
    'predict93_any_ai': 'int8',  # Using 93% threshold (high confidence)
    'predict93_ml': 'int8',
    'predict93_nlp': 'int8',
    'predict93_vision': 'int8',
    'predict93_kr': 'int8',
    'predict93_speech': 'int8',
    'predict93_planning': 'int8',
    'predict93_evo': 'int8',
    'predict93_hardware': 'int8'
}

aipd_cols = list(aipd_dtypes.keys()) + ['pub_dt']

aipd = pd.read_csv(
    'ai_model_predictions.csv',
    dtype=aipd_dtypes,
    usecols=aipd_cols,
    parse_dates=['pub_dt']
)

print(f"✓ Loaded {len(aipd):,} AI classification records")
print(f"  AI patents (93% threshold): {aipd['predict93_any_ai'].sum():,}")
print(f"  AI rate: {aipd['predict93_any_ai'].mean()*100:.2f}%")

aipd.head(3)

Loading USPTO AIPD AI classifications (2.8 GB)...
This may take 2-3 minutes...
✓ Loaded 15,437,542 AI classification records
  AI patents (93% threshold): 2,079,739
  AI rate: 13.47%


,doc_id,flag_patent,pub_dt,appl_id,predict93_any_ai,predict93_ml,predict93_evo,predict93_nlp,predict93_speech,predict93_vision,predict93_planning,predict93_kr,predict93_hardware
0,10000000,1,2018-06-19,14643719,0,0,0,0,0,0,0,0,0
1,10000001,1,2018-06-19,14962323,0,0,0,0,0,0,0,0,0
2,10000002,1,2018-06-19,15107519,1,0,0,0,0,0,1,0,0


### 1.4 Load CPC Codes (for validation)

In [6]:
# Load CPC codes from Task1 directory
print("Loading CPC codes for validation...")
print("Reading 3.1 GB file, this may take 3-5 minutes...")

cpc = pd.read_csv(
    '../Task1/g_cpc_current.tsv',
    sep='\t',
    dtype={'patent_id': str, 'cpc_group': str},
    usecols=['patent_id', 'cpc_group']
)

print(f"✓ Loaded {len(cpc):,} CPC code assignments")

# Filter to AI-related CPC codes (G06N*)
ai_cpc = cpc[cpc['cpc_group'].str.startswith('G06N', na=False)].copy()
ai_patent_ids_cpc = ai_cpc['patent_id'].unique()

print(f"  AI patents by CPC (G06N*): {len(ai_patent_ids_cpc):,}")

# Show sample CPC codes
print("\nSample AI CPC codes:")
print(ai_cpc['cpc_group'].value_counts().head(10))

Loading CPC codes for validation...
Reading 3.1 GB file, this may take 3-5 minutes...
✓ Loaded 57,969,447 CPC code assignments
  AI patents by CPC (G06N*): 95,567

Sample AI CPC codes:
cpc_group
G06N20/00     44741
G06N3/09      29190
G06N3/08      28354
G06N3/045     25239
G06N3/0464    23298
G06N7/01      12800
G06N3/084     10879
G06N3/044     10199
G06N5/04       9953
G06N3/04       8889
Name: count, dtype: int64


## Phase 2: Map Clinical Trial Firms to Patents

### 2.1 Get Clinical Trial Firm GVKEYs

In [7]:
# Extract unique GVKEYs from clinical trials
trial_gvkeys = clinical_trials['gvkey_sponsor'].dropna().unique()

print(f"Clinical trial firms: {len(trial_gvkeys):,} unique GVKEYs")
print(f"Sample GVKEYs: {trial_gvkeys[:5]}")

Clinical trial firms: 673 unique GVKEYs
Sample GVKEYs: [24454  8530  6730  1602 31628]


In [8]:
# FIX: Pad GVKEYs to 6 digits to match DISCERN format
print("\n" + "="*60)
print("⚠️  FIXING GVKEY FORMAT TO MATCH DISCERN")
print("="*60)
print(f"Before: Sample GVKEYs = {clinical_trials['gvkey_sponsor'].dropna().head().tolist()}")

# Convert to 6-digit zero-padded strings
clinical_trials['gvkey_sponsor'] = clinical_trials['gvkey_sponsor'].apply(
    lambda x: str(int(x)).zfill(6) if pd.notna(x) else x
)

# Update trial_gvkeys
trial_gvkeys = clinical_trials['gvkey_sponsor'].dropna().unique()

print(f"After:  Sample GVKEYs = {trial_gvkeys[:5]}")
print(f"✓ GVKEYs formatted as 6-digit strings (e.g., 1234 → '001234')")
print("="*60)


⚠️  FIXING GVKEY FORMAT TO MATCH DISCERN
Before: Sample GVKEYs = [24454, 8530, 6730, 24454, 24454]
After:  Sample GVKEYs = ['024454' '008530' '006730' '001602' '031628']
✓ GVKEYs formatted as 6-digit strings (e.g., 1234 → '001234')


### 2.2 Map GVKEY to PERMNO

In [9]:
# Map clinical trial GVKEYs to PERMNOs
print("Mapping GVKEYs to PERMNOs...")

trial_permno_mapping = permno_gvkey[permno_gvkey['gvkey'].isin(trial_gvkeys)].copy()

print(f"✓ Mapped {trial_permno_mapping['gvkey'].nunique():,} GVKEYs to PERMNOs")
print(f"  Total PERMNO mappings: {len(trial_permno_mapping):,} (includes time-varying mappings)")
print(f"  Match rate: {trial_permno_mapping['gvkey'].nunique() / len(trial_gvkeys) * 100:.1f}%")

trial_permno_mapping.head()

Mapping GVKEYs to PERMNOs...
✓ Mapped 344 GVKEYs to PERMNOs
  Total PERMNO mappings: 345 (includes time-varying mappings)
  Match rate: 51.1%


,gvkey,permno_adj,fyear1_adjust,fyearn_adjust,min_y_permno,max_y_permno
39,012180,10143,1986,2009,1986,2009
60,012181,10200,1986,2021,1986,2021
73,179598,10258,2008,2021,1986,2021
102,012250,10363,1986,2019,1986,2019
383,013365,11415,1987,2007,1987,2007


### 2.3 Filter DISCERN Patents to Clinical Trial Firms

In [10]:
# Get PERMNOs for clinical trial firms
trial_permnos = trial_permno_mapping['permno_adj'].unique()

print(f"Filtering patents to {len(trial_permnos):,} PERMNOs...")

# Filter DISCERN patents to clinical trial firms
firm_patents = discern_patents[discern_patents['permno_adj'].isin(trial_permnos)].copy()

print(f"✓ Found {len(firm_patents):,} patents from clinical trial firms")
print(f"  Unique firms: {firm_patents['permno_adj'].nunique():,}")
print(f"  Year range: {firm_patents['fyear'].min()}-{firm_patents['fyear'].max()}")

# Filter to 2000-2021 (align with clinical trial years)
firm_patents = firm_patents[
    (firm_patents['fyear'] >= 2000) & 
    (firm_patents['fyear'] <= 2021)
].copy()

print(f"\nAfter filtering to 2000-2021:")
print(f"  Patents: {len(firm_patents):,}")
print(f"  Firms: {firm_patents['permno_adj'].nunique():,}")

firm_patents.head()

Filtering patents to 338 PERMNOs...
✓ Found 83,167 patents from clinical trial firms
  Unique firms: 308
  Year range: 2000-2021

After filtering to 2000-2021:
  Patents: 83,167
  Firms: 308


,pgpub_id,application_id,filing_date,assignee_name,patent_id,fyear,published_date,name_std,id_name,sample,permno_adj,name_permno_adj
17,20020082676,9681080,2000-12-26,"SCIMED LIFE SYSTEMS, INC.",6398806,2000,2002-06-27,SCI MED LIFE SYS INC,8239,U,77605,BOSTON SCI CORP
26,20010044650,9681118,2001-01-12,"SCIMED LIFE SYSTEMS, INC.",NaN,2000,2001-11-22,SCI MED LIFE SYS INC,8239,U,77605,BOSTON SCI CORP
37,20020111671,9681191,2001-02-15,"SCIMED LIFE SYSTEMS, INC.",6540777,2000,2002-08-15,SCI MED LIFE SYS INC,8239,U,77605,BOSTON SCI CORP
38,20020108937,9681192,2001-02-15,"SCIMED LIFE SYSTEMS, INC.",6563080,2000,2002-08-15,SCI MED LIFE SYS INC,8239,U,77605,BOSTON SCI CORP
108,20020151966,9681394,2001-03-28,"SCIMED LIFE SYSTEMS, INC.",6585753,2000,2002-10-17,SCI MED LIFE SYS INC,8239,U,77605,BOSTON SCI CORP


### 2.4 Map PERMNO back to GVKEY (with year matching)

In [11]:
# Merge patents with PERMNO-GVKEY mapping
# Need to account for time-varying GVKEY assignments
print("Mapping patents to GVKEYs (with year matching)...")

firm_patents = firm_patents.merge(
    trial_permno_mapping[['gvkey', 'permno_adj', 'fyear1_adjust', 'fyearn_adjust']],
    on='permno_adj',
    how='left'
)

# Filter to valid GVKEY-year combinations
firm_patents = firm_patents[
    (firm_patents['fyear'] >= firm_patents['fyear1_adjust']) &
    (firm_patents['fyear'] <= firm_patents['fyearn_adjust'])
].copy()

print(f"✓ Mapped {len(firm_patents):,} patents to GVKEYs")
print(f"  Unique GVKEYs: {firm_patents['gvkey'].nunique():,}")

# Drop intermediate columns
firm_patents = firm_patents.drop(['fyear1_adjust', 'fyearn_adjust'], axis=1)

firm_patents.head()

Mapping patents to GVKEYs (with year matching)...
✓ Mapped 82,746 patents to GVKEYs
  Unique GVKEYs: 308


,pgpub_id,application_id,filing_date,assignee_name,patent_id,fyear,published_date,name_std,id_name,sample,permno_adj,name_permno_adj,gvkey
0,20020082676,9681080,2000-12-26,"SCIMED LIFE SYSTEMS, INC.",6398806,2000,2002-06-27,SCI MED LIFE SYS INC,8239,U,77605,BOSTON SCI CORP,025279
1,20010044650,9681118,2001-01-12,"SCIMED LIFE SYSTEMS, INC.",NaN,2000,2001-11-22,SCI MED LIFE SYS INC,8239,U,77605,BOSTON SCI CORP,025279
2,20020111671,9681191,2001-02-15,"SCIMED LIFE SYSTEMS, INC.",6540777,2000,2002-08-15,SCI MED LIFE SYS INC,8239,U,77605,BOSTON SCI CORP,025279
3,20020108937,9681192,2001-02-15,"SCIMED LIFE SYSTEMS, INC.",6563080,2000,2002-08-15,SCI MED LIFE SYS INC,8239,U,77605,BOSTON SCI CORP,025279
4,20020151966,9681394,2001-03-28,"SCIMED LIFE SYSTEMS, INC.",6585753,2000,2002-10-17,SCI MED LIFE SYS INC,8239,U,77605,BOSTON SCI CORP,025279


## Phase 3: AI Classification

### 3.1 Classify AI via USPTO AIPD (Primary Method)

In [12]:
# Merge with USPTO AIPD using patent_id
print("Classifying AI patents using USPTO AIPD...")

# Use patent_id (granted patents) for matching
firm_patents_aipd = firm_patents.merge(
    aipd[['doc_id', 'predict93_any_ai', 'predict93_ml', 'predict93_nlp', 
          'predict93_vision', 'predict93_kr', 'predict93_speech', 
          'predict93_planning', 'predict93_evo', 'predict93_hardware']],
    left_on='patent_id',
    right_on='doc_id',
    how='left'
)

# Create AI flag
firm_patents_aipd['is_ai_aipd'] = firm_patents_aipd['predict93_any_ai'].fillna(0).astype(bool)

print(f"✓ AIPD classification complete")
print(f"  Patents matched to AIPD: {firm_patents_aipd['doc_id'].notna().sum():,}")
print(f"  AI patents by AIPD: {firm_patents_aipd['is_ai_aipd'].sum():,}")
print(f"  AI rate: {firm_patents_aipd['is_ai_aipd'].mean()*100:.2f}%")

firm_patents_aipd.head()

Classifying AI patents using USPTO AIPD...
✓ AIPD classification complete
  Patents matched to AIPD: 45,813
  AI patents by AIPD: 1,069
  AI rate: 1.29%


,pgpub_id,application_id,filing_date,assignee_name,patent_id,fyear,published_date,name_std,id_name,sample,permno_adj,name_permno_adj,gvkey,doc_id,predict93_any_ai,predict93_ml,predict93_nlp,predict93_vision,predict93_kr,predict93_speech,predict93_planning,predict93_evo,predict93_hardware,is_ai_aipd
0,20020082676,9681080,2000-12-26,"SCIMED LIFE SYSTEMS, INC.",6398806,2000,2002-06-27,SCI MED LIFE SYS INC,8239,U,77605,BOSTON SCI CORP,025279,6398806,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False
1,20010044650,9681118,2001-01-12,"SCIMED LIFE SYSTEMS, INC.",NaN,2000,2001-11-22,SCI MED LIFE SYS INC,8239,U,77605,BOSTON SCI CORP,025279,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
2,20020111671,9681191,2001-02-15,"SCIMED LIFE SYSTEMS, INC.",6540777,2000,2002-08-15,SCI MED LIFE SYS INC,8239,U,77605,BOSTON SCI CORP,025279,6540777,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False
3,20020108937,9681192,2001-02-15,"SCIMED LIFE SYSTEMS, INC.",6563080,2000,2002-08-15,SCI MED LIFE SYS INC,8239,U,77605,BOSTON SCI CORP,025279,6563080,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False
4,20020151966,9681394,2001-03-28,"SCIMED LIFE SYSTEMS, INC.",6585753,2000,2002-10-17,SCI MED LIFE SYS INC,8239,U,77605,BOSTON SCI CORP,025279,6585753,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False


### 3.2 Classify AI via CPC Codes (Validation Method)

In [13]:
# Add CPC classification
print("Adding CPC G06N* classification for validation...")

firm_patents_aipd['is_ai_cpc'] = firm_patents_aipd['patent_id'].isin(ai_patent_ids_cpc)

print(f"✓ CPC classification complete")
print(f"  AI patents by CPC: {firm_patents_aipd['is_ai_cpc'].sum():,}")
print(f"  AI rate: {firm_patents_aipd['is_ai_cpc'].mean()*100:.2f}%")

Adding CPC G06N* classification for validation...
✓ CPC classification complete
  AI patents by CPC: 56
  AI rate: 0.07%


### 3.3 Combined AI Classification (Hybrid Approach)

In [14]:
# Create combined AI classification
def get_ai_method(row):
    """Determine AI classification method."""
    aipd = row['is_ai_aipd']
    cpc = row['is_ai_cpc']
    
    if aipd and cpc:
        return 'both'
    elif aipd:
        return 'aipd_only'
    elif cpc:
        return 'cpc_only'
    else:
        return None

firm_patents_aipd['ai_method'] = firm_patents_aipd.apply(get_ai_method, axis=1)
firm_patents_aipd['is_ai'] = firm_patents_aipd['is_ai_aipd'] | firm_patents_aipd['is_ai_cpc']

print("\n=== AI CLASSIFICATION SUMMARY ===")
print(f"Total patents: {len(firm_patents_aipd):,}")
print(f"\nAI Classification Breakdown:")
print(firm_patents_aipd['ai_method'].value_counts())

print(f"\nTotal AI patents: {firm_patents_aipd['is_ai'].sum():,}")
print(f"Overall AI rate: {firm_patents_aipd['is_ai'].mean()*100:.2f}%")

# Agreement rate
both_count = (firm_patents_aipd['ai_method'] == 'both').sum()
total_ai = firm_patents_aipd['is_ai'].sum()
agreement_rate = both_count / total_ai if total_ai > 0 else 0

print(f"\nClassification Agreement:")
print(f"  Both methods agree: {both_count:,} ({agreement_rate*100:.1f}% of AI patents)")


=== AI CLASSIFICATION SUMMARY ===
Total patents: 82,746

AI Classification Breakdown:
ai_method
aipd_only    1023
both           46
cpc_only       10
Name: count, dtype: int64

Total AI patents: 1,079
Overall AI rate: 1.30%

Classification Agreement:
  Both methods agree: 46 (4.3% of AI patents)


### 3.4 AI Component Technology Breakdown

In [15]:
# Analyze which AI technologies are most common
print("\n=== AI COMPONENT TECHNOLOGY BREAKDOWN ===")

ai_components = {
    'Machine Learning': 'predict93_ml',
    'Natural Language Processing': 'predict93_nlp',
    'Computer Vision': 'predict93_vision',
    'Knowledge Representation': 'predict93_kr',
    'Speech': 'predict93_speech',
    'Planning/Control': 'predict93_planning',
    'Evolutionary Computation': 'predict93_evo',
    'AI Hardware': 'predict93_hardware'
}

ai_patents_only = firm_patents_aipd[firm_patents_aipd['is_ai_aipd']]

for tech_name, col_name in ai_components.items():
    count = ai_patents_only[col_name].sum()
    pct = count / len(ai_patents_only) * 100 if len(ai_patents_only) > 0 else 0
    print(f"  {tech_name:30s}: {count:5,} ({pct:4.1f}%)")


=== AI COMPONENT TECHNOLOGY BREAKDOWN ===
  Machine Learning              : 142.0 (13.3%)
  Natural Language Processing   :  51.0 ( 4.8%)
  Computer Vision               : 257.0 (24.0%)
  Knowledge Representation      : 125.0 (11.7%)
  Speech                        :  32.0 ( 3.0%)
  Planning/Control              : 416.0 (38.9%)
  Evolutionary Computation      : 108.0 (10.1%)
  AI Hardware                   : 329.0 (30.8%)


## Phase 4: Aggregate to Firm-Year Level

### 4.1 Create Firm-Year Patent Metrics

In [16]:
# Aggregate to firm-year level
print("Aggregating to firm-year level...")

# Create aggregation
firm_year = firm_patents_aipd.groupby(['gvkey', 'fyear']).agg({
    'patent_id': 'count',  # total applications
    'is_ai': 'sum',  # AI applications (combined)
    'is_ai_aipd': 'sum',  # AI by AIPD
    'is_ai_cpc': 'sum',  # AI by CPC
}).reset_index()

# Rename columns
firm_year.columns = ['gvkey', 'year', 'total_applications', 
                     'ai_applications', 'ai_applications_aipd', 'ai_applications_cpc']

# Calculate 'both' method count
both_count_by_firm = firm_patents_aipd[firm_patents_aipd['ai_method'] == 'both'].groupby(
    ['gvkey', 'fyear']
).size().reset_index(name='ai_applications_both')

firm_year = firm_year.merge(both_count_by_firm, left_on=['gvkey', 'year'], 
                            right_on=['gvkey', 'fyear'], how='left')
firm_year['ai_applications_both'] = firm_year['ai_applications_both'].fillna(0).astype(int)
firm_year = firm_year.drop('fyear', axis=1, errors='ignore')

# Calculate derived metrics
firm_year['ai_share'] = firm_year['ai_applications'] / firm_year['total_applications']
firm_year['ai_dummy'] = (firm_year['ai_applications'] >= 1).astype(int)

# Sort
firm_year = firm_year.sort_values(['gvkey', 'year'])

print(f"✓ Created firm-year dataset")
print(f"  Observations: {len(firm_year):,}")
print(f"  Unique firms: {firm_year['gvkey'].nunique():,}")
print(f"  Year range: {firm_year['year'].min()}-{firm_year['year'].max()}")
print(f"  Firm-years with AI: {firm_year['ai_dummy'].sum():,} ({firm_year['ai_dummy'].mean()*100:.1f}%)")

firm_year.head(10)

Aggregating to firm-year level...
✓ Created firm-year dataset
  Observations: 2,831
  Unique firms: 308
  Year range: 2000-2021
  Firm-years with AI: 219 (7.7%)


,gvkey,year,total_applications,ai_applications,ai_applications_aipd,ai_applications_cpc,ai_applications_both,ai_share,ai_dummy
0,001259,2002,0,0,0,0,0,NaN,0
1,001259,2003,1,0,0,0,0,0.000000,0
2,001259,2007,3,0,0,0,0,0.000000,0
3,001259,2008,6,0,0,0,0,0.000000,0
4,001259,2009,0,0,0,0,0,NaN,0
5,001259,2010,0,0,0,0,0,NaN,0
6,001478,2000,31,1,1,0,0,0.032258,1
7,001478,2001,182,1,1,0,0,0.005495,1
8,001478,2002,154,1,1,0,0,0.006494,1
9,001478,2003,147,1,1,0,0,0.006803,1


### 4.2 Summary Statistics

In [17]:
# Summary statistics
print("\n=== FIRM-YEAR SUMMARY STATISTICS ===")
print(firm_year[[
    'total_applications', 'ai_applications', 'ai_applications_aipd',
    'ai_applications_cpc', 'ai_applications_both', 'ai_share'
]].describe())


=== FIRM-YEAR SUMMARY STATISTICS ===
       total_applications  ai_applications  ai_applications_aipd  \
count         2831.000000      2831.000000           2831.000000   
mean            16.186507         0.381137              0.377605   
std             56.339755         3.170522              3.145996   
min              0.000000         0.000000              0.000000   
25%              1.000000         0.000000              0.000000   
50%              3.000000         0.000000              0.000000   
75%              8.000000         0.000000              0.000000   
max            682.000000        66.000000             66.000000   

       ai_applications_cpc  ai_applications_both     ai_share  
count          2831.000000           2831.000000  2401.000000  
mean              0.019781              0.016249     0.008470  
std               0.258353              0.224987     0.049007  
min               0.000000              0.000000     0.000000  
25%               0.000000   

### 4.3 Temporal Trends

In [18]:
# Temporal trends
print("\n=== TEMPORAL TRENDS ===")

temporal = firm_year.groupby('year').agg({
    'total_applications': 'sum',
    'ai_applications': 'sum',
    'ai_share': 'mean',
    'ai_dummy': 'mean'
}).reset_index()

temporal['ai_share_overall'] = temporal['ai_applications'] / temporal['total_applications']

print(temporal)


=== TEMPORAL TRENDS ===
    year  total_applications  ai_applications  ai_share  ai_dummy  \
0   2000                 685               10  0.009206  0.098361   
1   2001                2160               27  0.008370  0.136986   
2   2002                1965               23  0.016378  0.162500   
3   2003                1999               18  0.004191  0.082353   
4   2004                1869               27  0.007344  0.104651   
5   2005                1717               21  0.007145  0.133333   
6   2006                2032               35  0.009291  0.111111   
7   2007                1726               35  0.016040  0.112150   
8   2008                1729               26  0.014560  0.091743   
9   2009                1770               31  0.010876  0.097087   
10  2010                1888               40  0.005762  0.063063   
11  2011                1964               55  0.007248  0.051282   
12  2012                2511               63  0.012209  0.095238   
13  2013 

## Phase 5: Export Results

### 5.1 Export Primary Deliverable: firm_year_patents_aipd.csv

In [19]:
# Export firm-year dataset
output_file = 'firm_year_patents_aipd.csv'
firm_year.to_csv(output_file, index=False)

print(f"✓ Exported: {output_file}")
print(f"  Size: {len(firm_year):,} rows × {len(firm_year.columns)} columns")
print(f"  Columns: {list(firm_year.columns)}")

✓ Exported: firm_year_patents_aipd.csv
  Size: 2,831 rows × 9 columns
  Columns: ['gvkey', 'year', 'total_applications', 'ai_applications', 'ai_applications_aipd', 'ai_applications_cpc', 'ai_applications_both', 'ai_share', 'ai_dummy']


### 5.2 Export Patent-Level Dataset (for validation)

In [20]:
# Export patent-level dataset
patent_level = firm_patents_aipd[[
    'patent_id', 'application_id', 'gvkey', 'fyear', 'assignee_name',
    'is_ai', 'is_ai_aipd', 'is_ai_cpc', 'ai_method',
    'predict93_ml', 'predict93_nlp', 'predict93_vision', 'predict93_kr'
]].copy()

output_file_patent = 'patent_level_aipd.csv'
patent_level.to_csv(output_file_patent, index=False)

print(f"✓ Exported: {output_file_patent}")
print(f"  Size: {len(patent_level):,} rows × {len(patent_level.columns)} columns")

✓ Exported: patent_level_aipd.csv
  Size: 82,746 rows × 13 columns


### 5.3 Merge with Clinical Trials (Optional)

In [21]:
# Aggregate clinical trials to firm-year
print("Merging with clinical trials data...")

trials_firm_year = clinical_trials.groupby(['gvkey_sponsor', 'start_year']).agg({
    'nct_id': 'count',
    'phase_number': 'mean'
}).reset_index()

trials_firm_year.columns = ['gvkey', 'year', 'num_trials', 'avg_phase']

# Merge
firm_year_merged = firm_year.merge(
    trials_firm_year,
    on=['gvkey', 'year'],
    how='outer'
)

# Fill NAs
firm_year_merged['num_trials'] = firm_year_merged['num_trials'].fillna(0).astype(int)
firm_year_merged = firm_year_merged.fillna(0)

# Export
output_file_merged = 'firm_year_merged_aipd.csv'
firm_year_merged.to_csv(output_file_merged, index=False)

print(f"✓ Exported: {output_file_merged}")
print(f"  Size: {len(firm_year_merged):,} rows × {len(firm_year_merged.columns)} columns")

firm_year_merged.head()

Merging with clinical trials data...
✓ Exported: firm_year_merged_aipd.csv
  Size: 4,373 rows × 11 columns


,gvkey,year,total_applications,ai_applications,ai_applications_aipd,ai_applications_cpc,ai_applications_both,ai_share,ai_dummy,num_trials,avg_phase
0,001259,2002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0
1,001259,2003,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0
2,001259,2007,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0
3,001259,2008,6.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0
4,001259,2009,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0


## Phase 6: Validation & Quality Checks

### 6.1 Classification Agreement Analysis

In [22]:
# Classification agreement matrix
print("\n=== CLASSIFICATION AGREEMENT ANALYSIS ===")

agreement_matrix = pd.crosstab(
    firm_patents_aipd['is_ai_aipd'],
    firm_patents_aipd['is_ai_cpc'],
    margins=True,
    rownames=['AIPD'],
    colnames=['CPC']
)

print(agreement_matrix)

# Agreement rate calculation
both_true = ((firm_patents_aipd['is_ai_aipd']) & (firm_patents_aipd['is_ai_cpc'])).sum()
aipd_only = ((firm_patents_aipd['is_ai_aipd']) & (~firm_patents_aipd['is_ai_cpc'])).sum()
cpc_only = ((~firm_patents_aipd['is_ai_aipd']) & (firm_patents_aipd['is_ai_cpc'])).sum()
total_ai = both_true + aipd_only + cpc_only

print(f"\nAgreement Rate: {both_true / total_ai * 100:.1f}%" if total_ai > 0 else "N/A")
print(f"AIPD detects more AI: {aipd_only / total_ai * 100:.1f}%" if total_ai > 0 else "N/A")
print(f"CPC detects more AI: {cpc_only / total_ai * 100:.1f}%" if total_ai > 0 else "N/A")


=== CLASSIFICATION AGREEMENT ANALYSIS ===
CPC    False  True    All
AIPD                     
False  81667    10  81677
True    1023    46   1069
All    82690    56  82746

Agreement Rate: 4.3%
AIPD detects more AI: 94.8%
CPC detects more AI: 0.9%


### 6.2 Top AI Firms

In [23]:
# Top firms by AI patents
print("\n=== TOP 10 FIRMS BY AI PATENTS (2015-2021) ===")

top_ai_firms = firm_year[
    firm_year['year'] >= 2015
].groupby('gvkey').agg({
    'ai_applications': 'sum',
    'total_applications': 'sum'
}).reset_index()

top_ai_firms['ai_share'] = top_ai_firms['ai_applications'] / top_ai_firms['total_applications']
top_ai_firms = top_ai_firms.sort_values('ai_applications', ascending=False).head(10)

print(top_ai_firms)


=== TOP 10 FIRMS BY AI PATENTS (2015-2021) ===
      gvkey  ai_applications  total_applications  ai_share
91   025279              341                3057  0.111547
6    008762              100                3035  0.032949
79   023812               26                 596  0.043624
2    003170               13                 590  0.022034
0    001602               10                 415  0.024096
195  133871                5                 111  0.045045
5    008530                5                 520  0.009615
177  062263                4                  86  0.046512
82   024040                3                 170  0.017647
10   013599                3                 326  0.009202


### 6.3 Sample AI Patents for Manual Review

In [24]:
# Sample AI patents for manual validation
print("\n=== SAMPLE AI PATENTS FOR MANUAL REVIEW ===")

sample_ai = firm_patents_aipd[firm_patents_aipd['is_ai']].sample(n=min(10, firm_patents_aipd['is_ai'].sum()))

print(sample_ai[[
    'patent_id', 'assignee_name', 'fyear', 'ai_method',
    'predict93_ml', 'predict93_nlp', 'predict93_vision'
]])


=== SAMPLE AI PATENTS FOR MANUAL REVIEW ===
      patent_id                                  assignee_name  fyear  \
61574  10391315  Boston Scientific Neuromodulation Corporation   2017   
49362   9289609  BOSTON SCIENTIFIC NEUROMODULATION CORPORATION   2014   
80847  11680930                Regeneron Pharmaceuticals, Inc.   2021   
58528  11627199                    SCHNEIDER ELECTRIC USA INC.   2016   
50527   9265960                       Cardiac Pacemakers, Inc.   2014   
73262  11273313  Boston Scientific Neuromodulation Corporation   2019   
10076   7331667                            BAUSCH & LOMB, INC.   2003   
55671   9669221  Boston Scientific Neuromodulation Corporation   2015   
45083   9510668                      COLGATE-PALMOLIVE COMPANY   2010   
19430   7619110                                 XenoPort, Inc.   2005   

       ai_method  predict93_ml  predict93_nlp  predict93_vision  
61574  aipd_only           0.0            0.0               0.0  
49362  aipd_only   

## Summary

### Key Results

This notebook successfully implemented the **DISCERN + USPTO AIPD hybrid approach** for AI patent classification:

1. **Firm-Patent Mapping:** Used DISCERN 2 to map clinical trial firms to their patent portfolios
2. **AI Classification:** Applied USPTO AIPD (BERT-based ML) as primary classifier
3. **Validation:** Cross-validated with CPC G06N* codes
4. **Outputs:** Generated firm-year aggregated datasets with three-tier AI classification

### Files Generated

1. **firm_year_patents_aipd.csv** - Primary deliverable (firm-year aggregated)
2. **patent_level_aipd.csv** - Patent-level classifications (for validation)
3. **firm_year_merged_aipd.csv** - Merged with clinical trials data

### Next Steps

1. Manual review of sample AI patents to validate classification quality
2. Regression analysis using firm_year_patents_aipd.csv
3. Sensitivity analysis comparing AIPD-only vs. CPC-only vs. both
4. Temporal analysis of AI adoption in biopharma (2000-2021)